# PD — Data Quality

Check duplicates, missing values, invalid values and potential leakage.


In [2]:
# ==========================================
# PD — Data Quality
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ruta del dataset
DATA_PATH = "../data/raw/cs-training.csv"

# Cargar dataset
df = pd.read_csv(DATA_PATH)

print("Dataset cargado correctamente.")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")


Dataset cargado correctamente.
Filas: 150,000
Columnas: 12


In [3]:
# ==========================================
# Revisión de duplicados e identificador
# ==========================================

print("Duplicados completos:")
print(f"Cantidad: {df.duplicated().sum():,}")

print("\nDuplicados según Unnamed: 0:")
print(f"Cantidad: {df['Unnamed: 0'].duplicated().sum():,}")

print("\nRango de Unnamed: 0:")
print(f"Mínimo: {df['Unnamed: 0'].min():,}")
print(f"Máximo: {df['Unnamed: 0'].max():,}")

print("\nCantidad de valores únicos:")
print(f"Valores únicos: {df['Unnamed: 0'].nunique():,}")

Duplicados completos:
Cantidad: 0

Duplicados según Unnamed: 0:
Cantidad: 0

Rango de Unnamed: 0:
Mínimo: 1
Máximo: 150,000

Cantidad de valores únicos:
Valores únicos: 150,000


In [4]:
# ==========================================
# Análisis de valores faltantes
# ==========================================

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
})

missing = missing.sort_values(
    "missing_count",
    ascending=False
)

display(missing)

,missing_count,missing_pct
MonthlyIncome,29731,19.820667
NumberOfDependents,3924,2.616000
Unnamed: 0,0,0.000000
SeriousDlqin2yrs,0,0.000000
age,0,0.000000
RevolvingUtilizationOfUnsecuredLines,0,0.000000
DebtRatio,0,0.000000
NumberOfTime30-59DaysPastDueNotWorse,0,0.000000
NumberOfOpenCreditLinesAndLoans,0,0.000000
NumberOfTimes90DaysLate,0,0.000000


In [5]:
# ==========================================
# Validación de edad
# ==========================================

age_checks = pd.DataFrame({
    "condition": [
        "age < 18",
        "age == 0",
        "age > 100"
    ],
    "count": [
        (df["age"] < 18).sum(),
        (df["age"] == 0).sum(),
        (df["age"] > 100).sum()
    ]
})

age_checks["percentage"] = (
    age_checks["count"] / len(df) * 100
)

display(age_checks)

,condition,count,percentage
0,age < 18,1,0.000667
1,age == 0,1,0.000667
2,age > 100,13,0.008667


In [6]:
# ==========================================
# Investigación de edades potencialmente inválidas
# ==========================================

age_invalid = (
    (df["age"] < 18) |
    (df["age"] > 100)
)

print(f"Registros con edad potencialmente inválida: {age_invalid.sum():,}")

print("\nDistribución del target:")
display(
    df.loc[age_invalid, "SeriousDlqin2yrs"]
      .value_counts()
      .sort_index()
)

print("\nRegistros afectados:")
display(
    df.loc[age_invalid].sort_values("age")
)

Registros con edad potencialmente inválida: 14

Distribución del target:


SeriousDlqin2yrs
0    13
1     1
Name: count, dtype: int64


Registros afectados:


,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
65695,65696,0,1.000000,0,1,0.436927,6000.0,6,0,2,0,2.0
7763,7764,0,0.069167,101,0,50.000000,NaN,2,0,0,0,0.0
116129,116130,1,0.002964,101,2,0.001387,2883.0,6,0,0,0,0.0
93813,93814,0,0.025780,101,0,0.013797,1666.0,5,0,0,0,0.0
90937,90938,0,0.000000,102,0,0.000000,NaN,12,0,0,0,0.0
96450,96451,0,0.109642,102,0,0.273844,3417.0,7,0,0,0,1.0
25561,25562,0,0.009866,102,0,0.002424,3300.0,3,0,0,0,0.0
57967,57968,0,0.001397,103,0,1798.500000,1.0,11,0,2,0,0.0
19884,19885,0,1.000000,103,0,0.000000,1600.0,3,0,0,0,0.0
135025,135026,0,0.004059,103,0,5.000000,NaN,6,0,0,0,0.0


In [7]:
(df["age"] < 18) | (df["age"] > 100)

0         False
1         False
2         False
3         False
4         False
          ...  
149995    False
149996    False
149997    False
149998    False
149999    False
Name: age, Length: 150000, dtype: bool

In [8]:
# ==========================================
# Resumen de edades potencialmente inválidas
# ==========================================

age_invalid_summary = pd.DataFrame({
    "condition": [
        "age = 0",
        "age 1-17",
        "age 101+"
    ],
    "count": [
        (df["age"] == 0).sum(),
        ((df["age"] >= 1) & (df["age"] < 18)).sum(),
        (df["age"] > 100).sum()
    ]
})

age_invalid_summary["percentage"] = (
    age_invalid_summary["count"] / len(df) * 100
)

display(age_invalid_summary)

,condition,count,percentage
0,age = 0,1,0.000667
1,age 1-17,0,0.000000
2,age 101+,13,0.008667


In [9]:
# ==========================================
# Validación de DebtRatio
# ==========================================

debt_ratio_checks = pd.DataFrame({
    "condition": [
        "DebtRatio < 0",
        "DebtRatio == 0",
        "DebtRatio > 1",
        "DebtRatio > 10",
        "DebtRatio > 100"
    ],
    "count": [
        (df["DebtRatio"] < 0).sum(),
        (df["DebtRatio"] == 0).sum(),
        (df["DebtRatio"] > 1).sum(),
        (df["DebtRatio"] > 10).sum(),
        (df["DebtRatio"] > 100).sum()
    ]
})

debt_ratio_checks["percentage"] = (
    debt_ratio_checks["count"] / len(df) * 100
)

display(debt_ratio_checks)

,condition,count,percentage
0,DebtRatio < 0,0,0.000000
1,DebtRatio == 0,4113,2.742000
2,DebtRatio > 1,35137,23.424667
3,DebtRatio > 10,28877,19.251333
4,DebtRatio > 100,24380,16.253333


In [10]:
# ==========================================
# Investigación de DebtRatio extremo
# ==========================================

debt_ratio_groups = pd.DataFrame({
    "group": [
        "DebtRatio <= 1",
        "DebtRatio > 1",
        "DebtRatio > 10",
        "DebtRatio > 100"
    ],
    "count": [
        (df["DebtRatio"] <= 1).sum(),
        (df["DebtRatio"] > 1).sum(),
        (df["DebtRatio"] > 10).sum(),
        (df["DebtRatio"] > 100).sum()
    ],
    "default_rate_pct": [
        df.loc[df["DebtRatio"] <= 1, "SeriousDlqin2yrs"].mean() * 100,
        df.loc[df["DebtRatio"] > 1, "SeriousDlqin2yrs"].mean() * 100,
        df.loc[df["DebtRatio"] > 10, "SeriousDlqin2yrs"].mean() * 100,
        df.loc[df["DebtRatio"] > 100, "SeriousDlqin2yrs"].mean() * 100
    ],
    "monthly_income_missing_pct": [
        df.loc[df["DebtRatio"] <= 1, "MonthlyIncome"].isna().mean() * 100,
        df.loc[df["DebtRatio"] > 1, "MonthlyIncome"].isna().mean() * 100,
        df.loc[df["DebtRatio"] > 10, "MonthlyIncome"].isna().mean() * 100,
        df.loc[df["DebtRatio"] > 100, "MonthlyIncome"].isna().mean() * 100
    ]
})

display(debt_ratio_groups)

,group,count,default_rate_pct,monthly_income_missing_pct
0,DebtRatio <= 1,114863,6.734109,1.590591
1,DebtRatio > 1,35137,6.520192,79.414862
2,DebtRatio > 10,28877,5.571908,92.706999
3,DebtRatio > 100,24380,5.783429,93.080394


In [11]:
# ==========================================
# Validación de utilización de crédito
# ==========================================

utilization_checks = pd.DataFrame({
    "condition": [
        "Utilization < 0",
        "Utilization == 0",
        "Utilization > 1",
        "Utilization > 2",
        "Utilization > 10",
        "Utilization > 100"
    ],
    "count": [
        (df["RevolvingUtilizationOfUnsecuredLines"] < 0).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] == 0).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] > 1).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] > 2).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] > 10).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] > 100).sum()
    ]
})

utilization_checks["percentage"] = (
    utilization_checks["count"] / len(df) * 100
)

display(utilization_checks)

,condition,count,percentage
0,Utilization < 0,0,0.000000
1,Utilization == 0,10878,7.252000
2,Utilization > 1,3321,2.214000
3,Utilization > 2,371,0.247333
4,Utilization > 10,241,0.160667
5,Utilization > 100,223,0.148667


In [12]:
# ==========================================
# Investigación de utilización extrema
# ==========================================

utilization_groups = pd.DataFrame({
    "group": [
        "Utilization <= 1",
        "Utilization > 1",
        "Utilization > 2",
        "Utilization > 10",
        "Utilization > 100"
    ],
    "count": [
        (df["RevolvingUtilizationOfUnsecuredLines"] <= 1).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] > 1).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] > 2).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] > 10).sum(),
        (df["RevolvingUtilizationOfUnsecuredLines"] > 100).sum()
    ],
    "default_rate_pct": [
        df.loc[
            df["RevolvingUtilizationOfUnsecuredLines"] <= 1,
            "SeriousDlqin2yrs"
        ].mean() * 100,

        df.loc[
            df["RevolvingUtilizationOfUnsecuredLines"] > 1,
            "SeriousDlqin2yrs"
        ].mean() * 100,

        df.loc[
            df["RevolvingUtilizationOfUnsecuredLines"] > 2,
            "SeriousDlqin2yrs"
        ].mean() * 100,

        df.loc[
            df["RevolvingUtilizationOfUnsecuredLines"] > 10,
            "SeriousDlqin2yrs"
        ].mean() * 100,

        df.loc[
            df["RevolvingUtilizationOfUnsecuredLines"] > 100,
            "SeriousDlqin2yrs"
        ].mean() * 100
    ]
})

display(utilization_groups)

,group,count,default_rate_pct
0,Utilization <= 1,146679,5.991996
1,Utilization > 1,3321,37.247817
2,Utilization > 2,371,14.555256
3,Utilization > 10,241,7.053942
4,Utilization > 100,223,4.932735


In [13]:
# ==========================================
# Investigación de valores especiales 96 y 98
# ==========================================

late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse"
]

special_values_summary = []

for col in late_cols:
    for value in [96, 98]:
        mask = df[col] == value

        special_values_summary.append({
            "variable": col,
            "special_value": value,
            "count": mask.sum(),
            "percentage": mask.mean() * 100,
            "default_rate_pct": (
                df.loc[mask, "SeriousDlqin2yrs"].mean() * 100
                if mask.sum() > 0
                else np.nan
            )
        })

special_values_summary = pd.DataFrame(
    special_values_summary
)

display(special_values_summary)

,variable,special_value,count,percentage,default_rate_pct
0,NumberOfTime30-59DaysPastDueNotWorse,96,5,0.003333,80.000000
1,NumberOfTime30-59DaysPastDueNotWorse,98,264,0.176000,54.166667
2,NumberOfTimes90DaysLate,96,5,0.003333,80.000000
3,NumberOfTimes90DaysLate,98,264,0.176000,54.166667
4,NumberOfTime60-89DaysPastDueNotWorse,96,5,0.003333,80.000000
5,NumberOfTime60-89DaysPastDueNotWorse,98,264,0.176000,54.166667


In [14]:
df[col] = df[col].replace({96: np.nan, 98: np.nan})

In [15]:
# ==========================================
# Coincidencia de valores especiales 96 y 98
# ==========================================

for value in [96, 98]:
    
    mask_special = (
        (df["NumberOfTime30-59DaysPastDueNotWorse"] == value) &
        (df["NumberOfTimes90DaysLate"] == value) &
        (df["NumberOfTime60-89DaysPastDueNotWorse"] == value)
    )
    
    print(
        f"Registros con {value} en las tres variables: "
        f"{mask_special.sum():,}"
    )
    
    if mask_special.sum() > 0:
        print(
            f"Default rate: "
            f"{df.loc[mask_special, 'SeriousDlqin2yrs'].mean() * 100:.2f}%"
        )
        
        display(
            df.loc[
                mask_special,
                late_cols + ["SeriousDlqin2yrs"]
            ].value_counts().reset_index(name="count")
        )

Registros con 96 en las tres variables: 0
Registros con 98 en las tres variables: 0


## Tratamiento de valores especiales 96 y 98

Las variables de historial de mora presentan los valores especiales 96 y 98:

- `NumberOfTime30-59DaysPastDueNotWorse`
- `NumberOfTimes90DaysLate`
- `NumberOfTime60-89DaysPastDueNotWorse`

La investigación exploratoria mostró que estos valores tienen una frecuencia muy baja respecto al total de observaciones y presentan tasas de default considerablemente diferentes al comportamiento de los valores habituales.

Además, los valores 96 y 98 no aparecen simultáneamente en las tres variables.

Dado que el diccionario de datos no proporciona una interpretación económica explícita para estos valores especiales, se tratarán como valores faltantes (`NaN`) en lugar de asignarles artificialmente un significado numérico.

El objetivo es evitar que estos códigos especiales sean interpretados por el modelo como cantidades reales de atrasos.

In [16]:
# ============================================
# Tratamiento de valores especiales 96 y 98
# ============================================

special_values = [96, 98]

for col in late_cols:
    df[col] = df[col].replace(special_values, np.nan)

print("Valores especiales reemplazados por NaN.")

Valores especiales reemplazados por NaN.


In [17]:
# Verificación posterior al tratamiento

special_check = pd.DataFrame({
    col: [
        (df[col] == 96).sum(),
        (df[col] == 98).sum(),
        df[col].isna().sum()
    ]
    for col in late_cols
}, index=["value_96", "value_98", "missing_after_treatment"])

display(special_check)

,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTimes90DaysLate,NumberOfTime60-89DaysPastDueNotWorse
value_96,0,0,0
value_98,0,0,0
missing_after_treatment,269,269,269


In [18]:
# ============================================
# Análisis de valores faltantes en MonthlyIncome
# ============================================

income_missing = df["MonthlyIncome"].isna()

income_missing_summary = pd.DataFrame({
    "group": [
        "MonthlyIncome disponible",
        "MonthlyIncome faltante"
    ],
    "count": [
        (~income_missing).sum(),
        income_missing.sum()
    ],
    "default_rate_pct": [
        df.loc[~income_missing, "SeriousDlqin2yrs"].mean() * 100,
        df.loc[income_missing, "SeriousDlqin2yrs"].mean() * 100
    ]
})

income_missing_summary["percentage"] = (
    income_missing_summary["count"] / len(df) * 100
)

display(income_missing_summary)

,group,count,default_rate_pct,percentage
0,MonthlyIncome disponible,120269,6.948590,80.179333
1,MonthlyIncome faltante,29731,5.613669,19.820667


In [19]:
# ============================================
# Análisis de valores faltantes en NumberOfDependents
# ============================================

dependents_missing = df["NumberOfDependents"].isna()

dependents_missing_summary = pd.DataFrame({
    "group": [
        "NumberOfDependents disponible",
        "NumberOfDependents faltante"
    ],
    "count": [
        (~dependents_missing).sum(),
        dependents_missing.sum()
    ],
    "default_rate_pct": [
        df.loc[~dependents_missing, "SeriousDlqin2yrs"].mean() * 100,
        df.loc[dependents_missing, "SeriousDlqin2yrs"].mean() * 100
    ]
})

dependents_missing_summary["percentage"] = (
    dependents_missing_summary["count"] / len(df) * 100
)

display(dependents_missing_summary)

,group,count,default_rate_pct,percentage
0,NumberOfDependents disponible,146076,6.741012,97.384
1,NumberOfDependents faltante,3924,4.561672,2.616


In [20]:
# ============================================
# Resumen actualizado de valores faltantes
# ============================================

missing_final = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
})

missing_final = missing_final.sort_values(
    "missing_count",
    ascending=False
)

display(missing_final)

,missing_count,missing_pct
MonthlyIncome,29731,19.820667
NumberOfDependents,3924,2.616000
NumberOfTime30-59DaysPastDueNotWorse,269,0.179333
NumberOfTimes90DaysLate,269,0.179333
NumberOfTime60-89DaysPastDueNotWorse,269,0.179333
RevolvingUtilizationOfUnsecuredLines,0,0.000000
SeriousDlqin2yrs,0,0.000000
Unnamed: 0,0,0.000000
age,0,0.000000
DebtRatio,0,0.000000


In [21]:
# ============================================
# Análisis de valores extremos: DebtRatio
# ============================================

debt_ratio_summary = pd.DataFrame({
    "metric": [
        "Mínimo",
        "P1",
        "P5",
        "P25",
        "Mediana",
        "P75",
        "P95",
        "P99",
        "Máximo"
    ],
    "value": [
        df["DebtRatio"].min(),
        df["DebtRatio"].quantile(0.01),
        df["DebtRatio"].quantile(0.05),
        df["DebtRatio"].quantile(0.25),
        df["DebtRatio"].median(),
        df["DebtRatio"].quantile(0.75),
        df["DebtRatio"].quantile(0.95),
        df["DebtRatio"].quantile(0.99),
        df["DebtRatio"].max()
    ]
})

display(debt_ratio_summary)

,metric,value
0,Mínimo,0.000000
1,P1,0.000000
2,P5,0.004329
3,P25,0.175074
4,Mediana,0.366508
5,P75,0.868254
6,P95,2449.000000
7,P99,4979.040000
8,Máximo,329664.000000


In [22]:
# ============================================
# DebtRatio según disponibilidad de MonthlyIncome
# ============================================

debt_income_summary = (
    df.assign(
        income_status=np.where(
            df["MonthlyIncome"].isna(),
            "MonthlyIncome faltante",
            "MonthlyIncome disponible"
        )
    )
    .groupby("income_status")["DebtRatio"]
    .agg(
        count="count",
        median="median",
        p75=lambda x: x.quantile(0.75),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        max="max"
    )
    .reset_index()
)

display(debt_income_summary)

,income_status,count,median,p75,p95,p99,max
0,MonthlyIncome disponible,120269,0.296023,0.482559,1.128771,661.82,61106.5
1,MonthlyIncome faltante,29731,1159.000000,2382.000000,4902.500000,8084.50,329664.0


In [24]:
# ============================================
# Análisis de valores extremos:
# RevolvingUtilizationOfUnsecuredLines
# ============================================

utilization_summary = pd.DataFrame({
    "metric": [
        "Mínimo",
        "P1",
        "P5",
        "P25",
        "Mediana",
        "P75",
        "P95",
        "P99",
        "Máximo"
    ],
    "value": [
        df["RevolvingUtilizationOfUnsecuredLines"].min(),
        df["RevolvingUtilizationOfUnsecuredLines"].quantile(0.01),
        df["RevolvingUtilizationOfUnsecuredLines"].quantile(0.05),
        df["RevolvingUtilizationOfUnsecuredLines"].quantile(0.25),
        df["RevolvingUtilizationOfUnsecuredLines"].median(),
        df["RevolvingUtilizationOfUnsecuredLines"].quantile(0.75),
        df["RevolvingUtilizationOfUnsecuredLines"].quantile(0.95),
        df["RevolvingUtilizationOfUnsecuredLines"].quantile(0.99),
        df["RevolvingUtilizationOfUnsecuredLines"].max()
    ]
})

display(utilization_summary)

,metric,value
0,Mínimo,0.000000
1,P1,0.000000
2,P5,0.000000
3,P25,0.029867
4,Mediana,0.154181
5,P75,0.559046
6,P95,1.000000
7,P99,1.092956
8,Máximo,50708.000000


In [25]:
# ============================================
# Utilización según disponibilidad de MonthlyIncome
# ============================================

util_income_summary = (
    df.assign(
        income_status=np.where(
            df["MonthlyIncome"].isna(),
            "MonthlyIncome faltante",
            "MonthlyIncome disponible"
        )
    )
    .groupby("income_status")["RevolvingUtilizationOfUnsecuredLines"]
    .agg(
        count="count",
        median="median",
        p75=lambda x: x.quantile(0.75),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        max="max"
    )
    .reset_index()
)

display(util_income_summary)

,income_status,count,median,p75,p95,p99,max
0,MonthlyIncome disponible,120269,0.177282,0.579428,1.0,1.089958,50708.0
1,MonthlyIncome faltante,29731,0.081697,0.440549,1.0,1.113603,22198.0


In [26]:
# ============================================
# Validación de valores negativos en variables de mora
# ============================================

late_negative_checks = pd.DataFrame({
    "variable": late_cols,
    "negative_count": [
        (df[col] < 0).sum()
        for col in late_cols
    ]
})

late_negative_checks["negative_pct"] = (
    late_negative_checks["negative_count"] / len(df) * 100
)

display(late_negative_checks)

,variable,negative_count,negative_pct
0,NumberOfTime30-59DaysPastDueNotWorse,0,0.0
1,NumberOfTimes90DaysLate,0,0.0
2,NumberOfTime60-89DaysPastDueNotWorse,0,0.0


In [27]:
# ============================================
# Distribución de valores en variables de mora
# ============================================

late_range_summary = pd.DataFrame({
    "variable": late_cols,
    "min": [
        df[col].min()
        for col in late_cols
    ],
    "p25": [
        df[col].quantile(0.25)
        for col in late_cols
    ],
    "median": [
        df[col].median()
        for col in late_cols
    ],
    "p75": [
        df[col].quantile(0.75)
        for col in late_cols
    ],
    "p95": [
        df[col].quantile(0.95)
        for col in late_cols
    ],
    "max": [
        df[col].max()
        for col in late_cols
    ]
})

display(late_range_summary)

,variable,min,p25,median,p75,p95,max
0,NumberOfTime30-59DaysPastDueNotWorse,0.0,0.0,0.0,0.0,2.0,13.0
1,NumberOfTimes90DaysLate,0.0,0.0,0.0,0.0,1.0,17.0
2,NumberOfTime60-89DaysPastDueNotWorse,0.0,0.0,0.0,0.0,0.0,11.0


In [28]:
# ============================================
# Consistencia entre variables de mora
# ============================================

late_presence = pd.DataFrame({
    "30_59": df["NumberOfTime30-59DaysPastDueNotWorse"].fillna(0) > 0,
    "60_89": df["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0) > 0,
    "90_plus": df["NumberOfTimes90DaysLate"].fillna(0) > 0
})

presence_summary = pd.DataFrame({
    "condition": [
        "Sin mora en ninguna variable",
        "Mora 30-59 días",
        "Mora 60-89 días",
        "Mora 90+ días",
        "Mora en 2 o más categorías"
    ],
    "count": [
        (~late_presence.any(axis=1)).sum(),
        late_presence["30_59"].sum(),
        late_presence["60_89"].sum(),
        late_presence["90_plus"].sum(),
        (late_presence.sum(axis=1) >= 2).sum()
    ]
})

presence_summary["percentage"] = (
    presence_summary["count"] / len(df) * 100
)

display(presence_summary)

,condition,count,percentage
0,Sin mora en ninguna variable,119906,79.937333
1,Mora 30-59 días,23713,15.808667
2,Mora 60-89 días,7335,4.890000
3,Mora 90+ días,8069,5.379333
4,Mora en 2 o más categorías,7318,4.878667


In [30]:
# ============================================
# Consistencia entre niveles de mora
# ============================================

late_consistency = pd.DataFrame({
    "condition": [
        "90+ > 60-89",
        "60-89 > 30-59",
        "90+ > 30-59"
    ],
    "count": [
        (
            df["NumberOfTimes90DaysLate"].fillna(0)
            > df["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0)
        ).sum(),

        (
            df["NumberOfTime60-89DaysPastDueNotWorse"].fillna(0)
            > df["NumberOfTime30-59DaysPastDueNotWorse"].fillna(0)
        ).sum(),

        (
            df["NumberOfTimes90DaysLate"].fillna(0)
            > df["NumberOfTime30-59DaysPastDueNotWorse"].fillna(0)
        ).sum()
    ]
})

late_consistency["percentage"] = (
    late_consistency["count"] / len(df) * 100
)

display(late_consistency)

,condition,count,percentage
0,90+ > 60-89,6452,4.301333
1,60-89 > 30-59,3769,2.512667
2,90+ > 30-59,4997,3.331333


# Conclusiones de Data Quality

La revisión de calidad permitió identificar y documentar los principales problemas del dataset:

1. **Duplicados:** no se encontraron registros duplicados completos ni duplicados en el identificador `Unnamed: 0`.

2. **Valores faltantes:**
   - `MonthlyIncome`: 29,731 registros (19.82%).
   - `NumberOfDependents`: 3,924 registros (2.62%).
   - Las variables de mora presentan 269 valores faltantes (0.18%) cada una después del tratamiento de los valores especiales 96 y 98.

3. **Valores especiales 96 y 98:**
   - Se identificaron en las tres variables de historial de mora.
   - Debido a que no se dispone de una interpretación económica explícita para estos códigos, fueron transformados a `NaN`.
   - No se eliminaron las observaciones.

4. **Edad:**
   - Se identificaron 14 registros potencialmente inválidos:
     - 1 registro con edad 0.
     - 13 registros con edad superior a 100.
   - Estos registros serán tratados posteriormente durante la preparación de variables.

5. **DebtRatio:**
   - Presenta una distribución altamente asimétrica y valores extremos.
   - Los valores extremos están fuertemente asociados con observaciones donde `MonthlyIncome` está faltante.
   - No se eliminaron ni limitaron los valores en esta etapa.

6. **RevolvingUtilizationOfUnsecuredLines:**
   - Presenta una distribución altamente asimétrica y valores extremos.
   - Se observó una relación no lineal entre utilización y default.
   - No se eliminaron los valores extremos en esta etapa.

7. **Variables de mora:**
   - No se encontraron valores negativos.
   - Los valores 96 y 98 fueron tratados como valores faltantes.
   - Las variables presentan una relación positiva general entre frecuencia de mora y default.
   - Se observaron combinaciones entre categorías de mora que no deben interpretarse automáticamente como errores de consistencia.

## Decisiones para la siguiente etapa

No se eliminarán registros de forma indiscriminada.

Las decisiones definitivas de imputación, transformación, tratamiento de valores extremos y creación de variables derivadas se realizarán en:

`03_feature_engineering.ipynb`

El objetivo será preparar las variables para el modelamiento de PD manteniendo la mayor cantidad posible de información y evitando introducir sesgos innecesarios.